# Phase 1 — Diffusion Autoencoder Training
**Continuous Sign Language Translation** · Phase 1 of 2

Trains a `SemanticEncoder` (1D-CNN) + `DiffusionDecoder` (1D-UNet) on 100 clips of the How2Sign landmark dataset (streaming from Hugging Face).  
The encoder learns a compact latent `Z [B, 15, 512]` that will be used in Phase 2.

**Runtime:** GPU (T4 or better) · **Est. time (100 clips, 3 epochs):** ~2 min


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
%pip install -q torch datasets huggingface_hub tqdm numpy


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────────────
# Option A (recommended): Colab Secrets  →  Runtime > Manage secrets > add HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)


## Model definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SemanticEncoder(nn.Module):
    """1D-CNN: [B, T=60, F=540] → [B, D=512, T'=15]"""
    def __init__(self, input_dim=540, latent_dim=512):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 256, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(256, 512, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(512, latent_dim, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(latent_dim)

    def forward(self, x):          # x: [B, T, F]
        x = x.transpose(1, 2)     # [B, F, T]
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return x                   # [B, D, T']


class DiffusionDecoder(nn.Module):
    """1D-UNet conditioned on Z; predicts noise."""
    def __init__(self, input_dim=540, latent_dim=512):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, 128), nn.GELU(), nn.Linear(128, 128)
        )
        self.upsample_z = nn.ConvTranspose1d(latent_dim, latent_dim, kernel_size=4, stride=4)
        self.net = nn.Sequential(
            nn.Conv1d(input_dim + latent_dim + 128, 512, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(512, 512, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(512, input_dim, kernel_size=3, padding=1),
        )

    def forward(self, x, z, t):
        x = x.transpose(1, 2)                                        # [B, F, T]
        t_emb = self.time_mlp(t.float()).unsqueeze(-1).expand(-1, -1, x.shape[-1])
        z_up  = self.upsample_z(z)
        out   = self.net(torch.cat([x, z_up, t_emb], dim=1))
        return out.transpose(1, 2)                                    # [B, T, F]


class TranslationModel(nn.Module):
    """FLAN-T5-small fed with latent Z [B, 15, 512] as encoder embeddings."""
    def __init__(self):
        super().__init__()
        from transformers import T5ForConditionalGeneration
        self.t5 = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

    def forward(self, z, labels=None):
        return self.t5(inputs_embeds=z, labels=labels) if labels is not None                else self.t5.generate(inputs_embeds=z)


## Dataset — streaming with on-the-fly feature engineering

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset

# 15 face landmarks capturing Non-Manual Markers (eyebrows, eye corners, nose, mouth)
FACE_LANDMARK_IDXS = [70, 105, 336, 300, 33, 133, 362, 263, 4, 61, 291, 13, 14, 17, 0]
_POSE_SLICE  = slice(0,   33)
_FACE_SLICE  = slice(33,  501)
_LHAND_SLICE = slice(501, 522)
_RHAND_SLICE = slice(522, 543)
T_WINDOW, T_STRIDE = 60, 30


def engineer_features(raw: np.ndarray):
    """raw: [T, 543, 3]  →  torch.Tensor [T, 540]"""
    T = raw.shape[0]
    if T < 2:
        return None
    pose  = raw[:, _POSE_SLICE,  :]   # [T, 33, 3]
    face  = raw[:, _FACE_SLICE,  :]   # [T, 468, 3]
    lhand = raw[:, _LHAND_SLICE, :]   # [T, 21, 3]
    rhand = raw[:, _RHAND_SLICE, :]   # [T, 21, 3]

    centre = pose.mean(axis=1, keepdims=True)  # body-centre per frame
    pose, face, lhand, rhand = (a - centre for a in (pose, face, lhand, rhand))

    kpts  = np.concatenate([pose, face[:, FACE_LANDMARK_IDXS, :], lhand, rhand], axis=1)  # [T,90,3]
    delta = np.zeros_like(kpts)
    delta[1:] = kpts[1:] - kpts[:-1]
    features = np.concatenate([kpts.reshape(T, -1), delta.reshape(T, -1)], axis=1)        # [T,540]
    return torch.from_numpy(features.astype(np.float32))


def sliding_windows(feat, window=T_WINDOW, stride=T_STRIDE):
    T = feat.shape[0]
    if T < window:
        yield F.pad(feat, (0, 0, 0, window - T))
    else:
        start = 0
        while start + window <= T:
            yield feat[start:start + window]
            start += stride


class RealSignLanguageDataset(IterableDataset):
    """Streams from bdanko/how2sign-landmarks-front-raw-parquet on Hugging Face."""
    def __init__(self, split="train", max_samples=None,
                 repo_id="bdanko/how2sign-landmarks-front-raw-parquet"):
        self.split, self.max_samples, self.repo_id = split, max_samples, repo_id

    def __iter__(self):
        from datasets import load_dataset
        ds = load_dataset(self.repo_id, split=self.split, streaming=True)
        count = 0
        for sample in ds:
            if self.max_samples and count >= self.max_samples:
                break
            raw = np.frombuffer(sample["features"], dtype=np.float32).reshape(sample["shape"])
            feat = engineer_features(raw)
            if feat is None:
                continue
            for chunk in sliding_windows(feat):
                yield chunk, sample.get("sentence", "")
            count += 1


## ⚙️ Configuration
Adjust these before running the training cell.


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
REPO_ID      = "bdanko/continuous-sign-language-translation"
SPLIT        = "train"
MAX_SAMPLES  = 100     # set to None for full dataset
BATCH_SIZE   = 32
LR           = 1e-4
EPOCHS       = 3
CKPT_DIR     = "/tmp/phase1_ckpt"


## Training loop

In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from tqdm.auto import tqdm

os.makedirs(CKPT_DIR, exist_ok=True)

encoder = SemanticEncoder().to(device)
decoder = DiffusionDecoder().to(device)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LR)
criterion = nn.MSELoss()

dataset = RealSignLanguageDataset(split=SPLIT, max_samples=MAX_SAMPLES)

for epoch in range(EPOCHS):
    total_loss, count = 0.0, 0
    batch_buf = []

    for features, _ in dataset:
        batch_buf.append(features)
        if len(batch_buf) < BATCH_SIZE:
            continue

        batch = torch.stack(batch_buf).to(device); batch_buf = []

        # Encode
        z = encoder(batch)                                      # [B, D, T']

        # Forward diffusion
        noise   = torch.randn_like(batch)
        t       = torch.randint(0, 1000, (batch.size(0), 1), device=device)
        alpha_t = (1 - t / 1000).view(-1, 1, 1)
        noisy   = alpha_t * batch + (1 - alpha_t) * noise

        # Denoise
        pred = decoder(noisy, z, t)
        loss = criterion(pred, batch)

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item(); count += 1

    avg = total_loss / max(count, 1)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg_loss={avg:.4f}")

print("Training complete.")


## Save & upload to Hugging Face

In [ ]:
import torch
from huggingface_hub import HfApi, create_repo, upload_folder

torch.save(encoder.state_dict(), f"{CKPT_DIR}/semantic_encoder.pth")
torch.save(decoder.state_dict(), f"{CKPT_DIR}/diffusion_decoder.pth")
print(f"Checkpoints saved to {CKPT_DIR}/")

api = HfApi()
create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True)
upload_folder(
    folder_path=CKPT_DIR,
    repo_id=REPO_ID,
    token=HF_TOKEN,
    commit_message=f"Phase 1 checkpoint — {EPOCHS} epochs, {MAX_SAMPLES} clips",
)
print(f"Uploaded to https://huggingface.co/{REPO_ID} ✓")
